# PPO Continuous State and Reward Lab

This notebook runs PPO continuous baselines while keeping the editable experiment definition in a few visible cells. The intended edit points are the state-feature list, the reward weights/scales, and the usual training knobs. The training code itself stays in the package and the notebook writes JSON spec files so each run can be repeated later.


## 1. Setup

Run this first. It finds the project root, imports the notebook helpers, and loads the state/reward spec builders used by the subprocess runner.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for root in (candidate, candidate / 'TeleopWithRL'):
        if (root / 'matlab_literal_env').exists() and (root / 'notebooks' / '_teleop_nb.py').exists():
            for path_to_add in (root.parent, root):
                if str(path_to_add) not in sys.path:
                    sys.path.insert(0, str(path_to_add))
            break
    else:
        continue
    break
else:
    raise RuntimeError('Could not find TeleopWithRL notebook root.')

from notebooks._teleop_nb import load_json, project_python_executable, repo_paths, show_image, show_rows
from TeleopWithRL import config as cfg
from TeleopWithRL.matlab_literal_env.policy_gradient_experiments.paths import suite_root as policy_gradient_suite_root
from TeleopWithRL.matlab_literal_env.studies.common import save_json
from TeleopWithRL.matlab_literal_env.studies.dqn_state_variants import (
    available_custom_state_feature_rows,
    build_custom_dqn_state_variant_from_spec,
    get_dqn_state_variant,
)
from TeleopWithRL.matlab_literal_env.studies.rewarding import (
    DEFAULT_ACTION_DELTA_SCALE_V,
    DEFAULT_ACTION_SCALE_V,
    DEFAULT_FORCE_DIFF_SCALE_N,
    DEFAULT_TRACKING_SCALE_M,
    DEFAULT_TRANSPARENCY_SCALE_W,
    DEFAULT_VELOCITY_ERROR_SCALE_MPS,
    compute_reward_terms,
    reward_formula_from_context,
    reward_variant_from_name,
    reward_variant_from_spec,
)

P = repo_paths()
REPO = P['repo']
WORKSPACE = REPO.parent
PYTHON = project_python_executable(REPO)
PG_RESULTS = REPO / 'matlab_literal_env' / 'policy_gradient_experiments' / 'results'

def short_hash(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode('utf-8')
    return hashlib.sha1(raw).hexdigest()[:8]


## 2. Experiment Knobs

These are the normal run settings. For quick smoke tests, reduce `train_episodes`, `parallel_envs`, and `test_episodes`; for serious comparisons, keep the seed fixed and change one idea at a time.


In [ ]:
ALGO_KEY = 'ppo_continuous'
ALGO_LABEL = 'PPO Continuous'
ALGO_TAG = 'ppo'
RUN_DIR = 'ppo'
FINAL_RESULTS_SUBFOLDER = 'ppoFinalModel'

CFG = {
    'experiment_label': 'state_reward_lab01',
    'env_mode': 'changing_skin_fat',
    'episode_duration_s': 30.0,
    'env_switch_time_s': 10.0,
    'reset_position_mode': 'midpoint',
    'stroke_limit_mode': 'clamp',
    'force_amp_N': 5.0,
    'force_bias_N': 15.0,
    'force_freq_rad_s': 6.0,
    'force_phase_rad': 0.0,
    'force_waveform': 'sine',
    'train_episodes': 334,
    'total_timesteps': 500_000,
    'parallel_envs': 8,
    'vec_env': 'subproc',
    'ppo_n_steps': 256,
    'ppo_batch_size': 512,
    'ppo_n_epochs': 4,
    'ppo_device': 'auto',
    'eval_every_episodes': 150,
    'test_episodes': 32,
    'seed': 42,
    'parallel_workers': 1,
    'worker_torch_threads': 1,
    'skip_existing': False,
}

FE_MODE = 'switched_dynamics'
FE_KEY = 'dyn'
MODE_LABELS = {
    FE_KEY: FE_MODE,
}


## 3. State Space

This is the main state-space cell. The default is a full physical MDP-style state: master/slave positions, velocities, chamber pressures, line mass flows, valve states, forces, previous action, and time/context. Change `True` to `False` to remove a variable, or turn on one of the derived variables near the bottom.


In [ ]:
USE_STATE_FEATURE = {
    # Physical plant state: positions and velocities
    'x_m': True,
    'x_s': True,
    'v_m': True,
    'v_s': True,

    # Physical plant state: chamber pressures
    'P_m1': False,
    'P_m2': False,
    'P_s1': False,
    'P_s2': False,

    # Physical plant state: tube mass-flow states
    'mdot_L1': False,
    'mdot_L2': False,

    # Physical plant state: valve dynamics
    'x_v': False,
    'x_v_dot': False,

    # Exogenous inputs, measured forces, and context
    'F_h': False,
    'F_e': False,
    'u_v': False,
    'env_id': False,
    'time_fraction': False,

    # Optional action/context variables
    'requested_u_v': False,

    # Optional derived variables. These are redundant if the raw variables above are present,
    # but they can help smaller networks by giving common errors directly.
    'tracking_error': False,
    'velocity_error': False,
    'transparency_error': False,
    'force_diff': False,
    'delta_P_m': False,
    'delta_P_s': False,
    'P_m1_minus_P_s1': False,
    'P_m2_minus_P_s2': False,

    # Optional equilibrium-centered positions. Usually choose these OR absolute x_m/x_s.
    'x_m_eq': False,
    'x_s_eq': False,
    'x_m_centered': False,
    'x_s_centered': False,
}

ALL_STATE_VARIABLES = available_custom_state_feature_rows()
state_lookup = {row['feature']: row for row in ALL_STATE_VARIABLES}
unknown_features = [feature for feature in USE_STATE_FEATURE if feature not in state_lookup]
if unknown_features:
    raise KeyError(f'Unknown state feature(s): {unknown_features}')

def state_feature_group(feature):
    if feature in {'x_m', 'x_s', 'x_m_eq', 'x_s_eq', 'x_m_centered', 'x_s_centered', 'tracking_error'}:
        return 'position'
    if feature in {'v_m', 'v_s', 'velocity_error'}:
        return 'velocity'
    if feature in {'P_m1', 'P_m2', 'P_s1', 'P_s2', 'delta_P_m', 'delta_P_s', 'P_m1_minus_P_s1', 'P_m2_minus_P_s2'}:
        return 'pressure'
    if feature in {'mdot_L1', 'mdot_L2'}:
        return 'mass_flow'
    if feature in {'x_v', 'x_v_dot'}:
        return 'valve'
    if feature in {'F_h', 'F_e', 'force_diff', 'transparency_error'}:
        return 'force_transparency'
    if feature in {'u_v', 'requested_u_v'}:
        return 'action_memory'
    if feature in {'env_id', 'time_fraction'}:
        return 'context'
    return 'other'

catalog_rows = []
for row in ALL_STATE_VARIABLES:
    feature = row['feature']
    display_row = dict(row)
    display_row['group'] = state_feature_group(feature)
    display_row['selected'] = bool(USE_STATE_FEATURE.get(feature, False))
    catalog_rows.append(display_row)
show_rows(catalog_rows, title='Full MDP variable catalog', max_rows=80)

STATE_VARIANT = get_dqn_state_variant('S11_absolute_posvel_only')
SELECTED_STATE_FEATURES = list(STATE_VARIANT.feature_names)

STATE_SPEC = {
    'name': STATE_VARIANT.name,
    'description': STATE_VARIANT.description,
    'selected_features': SELECTED_STATE_FEATURES,
}

selected_state_rows = []
for idx, feature in enumerate(STATE_VARIANT.feature_names, start=1):
    row = dict(state_lookup[feature])
    row['order'] = idx
    row['group'] = state_feature_group(feature)
    selected_state_rows.append(row)

show_rows(selected_state_rows, title=f'Selected state space: {STATE_VARIANT.name}', max_rows=80)
print(f'Observation dimension: {STATE_VARIANT.obs_dim}')


## 4. Action Space

PPO continuous keeps the same action interface: one continuous valve-voltage command clipped to the environment voltage range.


In [ ]:
ACTION_SPACE = {
    'action': 'u_v',
    'type': 'continuous voltage',
    'low_v': float(np.min(cfg.V_LEVELS)),
    'high_v': float(np.max(cfg.V_LEVELS)),
    'scale_used_for_state_and_reward': DEFAULT_ACTION_SCALE_V,
}
show_rows([ACTION_SPACE], title='Action space used by PPO continuous', max_rows=5)


## 5. Reward Function

This cell controls the **structure** and **normalization** of the reward. Each row in `REWARD_TERMS` chooses what signal to use, how to shape it, and which named scale normalizes it. To change normalization, edit `scale_name` or add a new entry to `REWARD_SCALE_CATALOG`.


In [ ]:
VALVE_POSITION_SCALE = max(abs(float(cfg.KV)) * DEFAULT_ACTION_SCALE_V, 1e-9)
VALVE_VELOCITY_SCALE = max(150.0 * VALVE_POSITION_SCALE, 1e-9)

REWARD_SOURCE_CATALOG = [
    {'source': 'pos_error', 'meaning': 'x_m - x_s tracking error [m]', 'suggested_scale': 'tracking_error_limit_m'},
    {'source': 'velocity_error', 'meaning': 'v_m - v_s [m/s]', 'suggested_scale': 'velocity_error_mps'},
    {'source': 'transparency_error', 'meaning': '(F_e * v_m) - (F_h * v_s) [W]', 'suggested_scale': 'power_error_practical_w'},
    {'source': 'force_diff', 'meaning': 'F_e - F_h [N]', 'suggested_scale': 'force_difference_n'},
    {'source': 'u_v', 'meaning': 'applied valve voltage [V]', 'suggested_scale': 'action_voltage_v'},
    {'source': 'action_delta', 'meaning': 'u_v - previous_u_v [V]', 'suggested_scale': 'action_delta_voltage_v'},
    {'source': 'F_h', 'meaning': 'human/master force input [N]', 'suggested_scale': 'human_force_est_n'},
    {'source': 'F_e', 'meaning': 'environment/slave force [N]', 'suggested_scale': 'environment_force_theoretical_n'},
    {'source': 'x_m', 'meaning': 'absolute master position [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'x_s', 'meaning': 'absolute slave position [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'x_m_centered', 'meaning': 'master position relative to reset equilibrium [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'x_s_centered', 'meaning': 'slave position relative to reset equilibrium [m]', 'suggested_scale': 'half_stroke_m'},
    {'source': 'v_m', 'meaning': 'master velocity [m/s]', 'suggested_scale': 'obs_velocity_mps'},
    {'source': 'v_s', 'meaning': 'slave velocity [m/s]', 'suggested_scale': 'obs_velocity_mps'},
    {'source': 'P_m1', 'meaning': 'master chamber 1 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'P_m2', 'meaning': 'master chamber 2 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'P_s1', 'meaning': 'slave chamber 1 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'P_s2', 'meaning': 'slave chamber 2 pressure [Pa]', 'suggested_scale': 'pressure_supply_pa'},
    {'source': 'delta_P_m', 'meaning': 'P_m1 - P_m2 [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'delta_P_s', 'meaning': 'P_s1 - P_s2 [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'P_m1_minus_P_s1', 'meaning': 'cross-piston chamber 1 pressure error [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'P_m2_minus_P_s2', 'meaning': 'cross-piston chamber 2 pressure error [Pa]', 'suggested_scale': 'pressure_difference_pa'},
    {'source': 'mdot_L1', 'meaning': 'tube mass-flow state 1 [kg/s]', 'suggested_scale': 'mass_flow_kg_s'},
    {'source': 'mdot_L2', 'meaning': 'tube mass-flow state 2 [kg/s]', 'suggested_scale': 'mass_flow_kg_s'},
    {'source': 'x_v', 'meaning': 'valve spool position', 'suggested_scale': 'valve_position'},
    {'source': 'x_v_dot', 'meaning': 'valve spool velocity', 'suggested_scale': 'valve_velocity'},
    {'source': 'edge_severity', 'meaning': '0 away from stroke edge, 1 at edge; uses edge_buffer_m', 'suggested_scale': 'unit_interval'},
    {'source': 'low_force_edge_severity', 'meaning': 'edge severity multiplied by low-force factor', 'suggested_scale': 'unit_interval'},
    {'source': 'time', 'meaning': 'episode time [s]', 'suggested_scale': 'episode_duration_s'},
    {'source': 'time_fraction', 'meaning': 'episode progress from 0 to 1', 'suggested_scale': 'unit_interval'},
    {'source': 'env_id', 'meaning': 'skin=0, fat=1', 'suggested_scale': 'unit_interval'},
]

REWARD_SCALE_CATALOG = {
    # Dimensionless/context scales
    'unit_interval': {'value': 1.0, 'unit': '-', 'use_for': 'edge_severity, time_fraction, env_id'},
    'episode_duration_s': {'value': float(CFG['episode_duration_s']), 'unit': 's', 'use_for': 'time'},
    'env_switch_time_s': {'value': float(CFG['env_switch_time_s']), 'unit': 's', 'use_for': 'pre/post switch timing terms'},

    # Position scales
    'stroke_m': {'value': float(cfg.L_CYL), 'unit': 'm', 'use_for': 'absolute x_m/x_s over full cylinder stroke'},
    'half_stroke_m': {'value': float(cfg.OBS_SCALE_POS), 'unit': 'm', 'use_for': 'centered positions'},
    'tracking_error_limit_m': {'value': float(cfg.MAX_POSITION_ERROR), 'unit': 'm', 'use_for': 'pos_error / tracking_error'},
    'tracking_failure_threshold_m': {'value': float(cfg.POS_ERROR_FAIL_THRESHOLD), 'unit': 'm', 'use_for': 'large tracking-error safety terms'},
    'reference_position_amp_m': {'value': float(cfg.REF_POS_AMP), 'unit': 'm', 'use_for': 'reference-style position terms'},
    'two_mm_m': {'value': 0.002, 'unit': 'm', 'use_for': 'small tracking deadbands or margins'},
    'five_mm_m': {'value': 0.005, 'unit': 'm', 'use_for': 'tracking tolerance bonuses'},
    'one_cm_m': {'value': 0.010, 'unit': 'm', 'use_for': 'moderate tracking tolerance'},

    # Velocity scales
    'obs_velocity_mps': {'value': float(cfg.OBS_SCALE_VEL), 'unit': 'm/s', 'use_for': 'v_m, v_s'},
    'velocity_error_mps': {'value': DEFAULT_VELOCITY_ERROR_SCALE_MPS, 'unit': 'm/s', 'use_for': 'velocity_error'},
    'max_geometric_velocity_mps': {'value': float(cfg.V_MAX_GEOM), 'unit': 'm/s', 'use_for': 'large velocity safety terms'},
    'reference_velocity_mps': {'value': float(2.0 * np.pi * cfg.REF_POS_FREQ * cfg.REF_POS_AMP), 'unit': 'm/s', 'use_for': 'reference-style velocities'},
    'force_velocity_mps': {'value': float(cfg.FORCE_INPUT_AMP / max(cfg.BETA, 1e-9)), 'unit': 'm/s', 'use_for': 'force-driven velocity scale'},

    # Power/transparency scales
    'power_error_practical_w': {'value': float(cfg.MAX_POWER_ERROR), 'unit': 'W', 'use_for': 'transparency_error'},
    'power_error_theoretical_w': {'value': float(cfg.MAX_POWER_ERROR_THEORETICAL), 'unit': 'W', 'use_for': 'conservative transparency_error'},

    # Force scales
    'force_input_amp_n': {'value': float(cfg.FORCE_INPUT_AMP), 'unit': 'N', 'use_for': 'F_h sinusoid amplitude'},
    'human_force_est_n': {'value': float(cfg.F_H_SCALE_EST), 'unit': 'N', 'use_for': 'F_h'},
    'environment_force_theoretical_n': {'value': float(cfg.F_E_MAX_THEORETICAL), 'unit': 'N', 'use_for': 'F_e'},
    'force_difference_n': {'value': DEFAULT_FORCE_DIFF_SCALE_N, 'unit': 'N', 'use_for': 'force_diff'},

    # Pressure and flow scales
    'pressure_supply_pa': {'value': float(cfg.P_SUPPLY), 'unit': 'Pa', 'use_for': 'raw chamber pressures'},
    'pressure_atmosphere_pa': {'value': float(cfg.P_ATM), 'unit': 'Pa', 'use_for': 'absolute pressure offsets'},
    'pressure_difference_pa': {'value': float(cfg.OBS_SCALE_PRESSURE), 'unit': 'Pa', 'use_for': 'pressure differences'},
    'mass_flow_kg_s': {'value': float(cfg.OBS_SCALE_FLOW), 'unit': 'kg/s', 'use_for': 'mdot_L1, mdot_L2'},

    # Action and valve scales
    'action_voltage_v': {'value': DEFAULT_ACTION_SCALE_V, 'unit': 'V', 'use_for': 'u_v'},
    'action_delta_voltage_v': {'value': DEFAULT_ACTION_DELTA_SCALE_V, 'unit': 'V', 'use_for': 'action_delta / smoothness'},
    'valve_position': {'value': VALVE_POSITION_SCALE, 'unit': '-', 'use_for': 'x_v'},
    'valve_velocity': {'value': VALVE_VELOCITY_SCALE, 'unit': '1/s', 'use_for': 'x_v_dot'},
}

show_rows(REWARD_SOURCE_CATALOG, title='Reward sources you can use', max_rows=80)
show_rows([
    {'scale_name': name, **payload}
    for name, payload in REWARD_SCALE_CATALOG.items()
], title='Reward normalization scales you can use', max_rows=80)

REWARD_SHAPES = [
    {'shape': 'square', 'formula': '((source - target) / scale)^2'},
    {'shape': 'absolute', 'formula': 'abs((source - target) / scale)'},
    {'shape': 'deadband_square', 'formula': 'max(abs(source - target) - deadband, 0)^2 / scale^2'},
    {'shape': 'deadband_abs', 'formula': 'max(abs(source - target) - deadband, 0) / scale'},
    {'shape': 'above_threshold_square', 'formula': 'max(source - threshold, 0)^2 / scale^2'},
    {'shape': 'below_threshold_square', 'formula': 'max(threshold - source, 0)^2 / scale^2'},
    {'shape': 'tolerance_bonus', 'formula': 'max(0, 1 - abs(source - target) / margin)'},
    {'shape': 'gaussian_bonus', 'formula': 'exp(-0.5 * ((source - target) / scale)^2)'},
]
show_rows(REWARD_SHAPES, title='Reward shapes you can use', max_rows=20)

REWARD_TERMS = [
    {
        'name': 'tracking',
        'source': 'pos_error',
        'shape': 'square',
        'sign': 'penalty',
        'weight': 40.0,
        'scale_name': 'tracking_error_limit_m',
    },
    {
        'name': 'smooth_action',
        'source': 'action_delta',
        'shape': 'square',
        'sign': 'penalty',
        'weight': 0.05,
        'scale_name': 'action_delta_voltage_v',
    },
    # Example bonus: uncomment to reward very small tracking error directly.
    # {
    #     'name': 'near_zero_tracking_bonus',
    #     'source': 'pos_error',
    #     'shape': 'tolerance_bonus',
    #     'sign': 'bonus',
    #     'weight': 0.5,
    #     'target': 0.0,
    #     'margin_name': 'five_mm_m',
    #     'scale_name': 'five_mm_m',
    # },
    # Example pressure regularizer: uncomment to discourage large master pressure imbalance.
    # {
    #     'name': 'master_pressure_balance',
    #     'source': 'delta_P_m',
    #     'shape': 'absolute',
    #     'sign': 'penalty',
    #     'weight': 0.1,
    #     'scale_name': 'pressure_difference_pa',
    # },
]

REWARD_VARIANT = reward_variant_from_name('track_jerk_no_force_trans')
REWARD_SPEC = {
    'name': REWARD_VARIANT.name,
    'description': 'Built-in tracking + action-smoothness reward. Force and transparency penalties are removed.',
    'builtin_variant': REWARD_VARIANT.name,
    'scale_catalog': REWARD_SCALE_CATALOG,
    'terms': REWARD_TERMS,
    'penalties': {
        'stroke_limit': 250.0,
        'invalid_state': 100.0,
        'tracking_error_fail': 1000.0,
        'edge_buffer_m': 0.0,
        'low_force_threshold_n': 0.0,
    },
}

reward_rows = []
for term in REWARD_TERMS:
    reward_rows.append({
        'name': term['name'],
        'source': term['source'],
        'shape': term['shape'],
        'sign': term['sign'],
        'weight': term['weight'],
        'scale_name': term.get('scale_name', ''),
        'scale_value': REWARD_SCALE_CATALOG.get(term.get('scale_name', ''), {}).get('value', ''),
        'target': term.get('target', 0.0),
        'deadband': term.get('deadband_name', term.get('deadband', 0.0)),
        'threshold': term.get('threshold', 0.0),
        'margin': term.get('margin', ''),
    })
show_rows(reward_rows, title=f'Reward formula: {REWARD_VARIANT.name}', max_rows=40)


## 6. Reward Sanity Check

This cell evaluates the reward formula on hand-made scenarios before training. If a bad scenario scores better than a good one, fix the reward before launching PPO.


In [ ]:
def reward_context(pos_error_m, transparency_error_w, velocity_error_mps=0.0, force_diff_n=0.0, u_v=0.0, action_delta_v=0.0):
    f_h = 10.0
    f_e = f_h + force_diff_n
    v_s = 0.0
    v_m = velocity_error_mps
    x_s = 0.5 * float(cfg.L_CYL)
    x_m = x_s + pos_error_m
    context = {
        'time': 0.0,
        'time_fraction': 0.0,
        'env_id': 0.0,
        'x_m': x_m,
        'x_s': x_s,
        'x_m_centered': pos_error_m,
        'x_s_centered': 0.0,
        'v_m': v_m,
        'v_s': v_s,
        'P_m1': float(cfg.P_SUPPLY),
        'P_m2': float(cfg.P_SUPPLY),
        'P_s1': float(cfg.P_SUPPLY),
        'P_s2': float(cfg.P_SUPPLY),
        'delta_P_m': 0.0,
        'delta_P_s': 0.0,
        'P_m1_minus_P_s1': 0.0,
        'P_m2_minus_P_s2': 0.0,
        'mdot_L1': 0.0,
        'mdot_L2': 0.0,
        'x_v': 0.0,
        'x_v_dot': 0.0,
        'F_h': f_h,
        'F_e': f_e,
        'u_v': u_v,
        'requested_u_v': u_v,
        'action_delta': action_delta_v,
        'pos_error': pos_error_m,
        'tracking_error': pos_error_m,
        'velocity_error': velocity_error_mps,
        'transparency_error': transparency_error_w,
        'force_diff': force_diff_n,
        'edge_severity': 0.0,
        'low_force_edge_severity': 0.0,
    }
    for key, value in list(context.items()):
        context[f'abs_{key}'] = abs(float(value))
    return context

def reward_case(name, pos_error_m, transparency_error_w, velocity_error_mps=0.0, force_diff_n=0.0, u_v=0.0, action_delta_v=0.0):
    context = reward_context(
        pos_error_m,
        transparency_error_w,
        velocity_error_mps=velocity_error_mps,
        force_diff_n=force_diff_n,
        u_v=u_v,
        action_delta_v=action_delta_v,
    )
    if REWARD_VARIANT.formula_terms:
        reward, grouped_terms, custom_terms = reward_formula_from_context(context, REWARD_VARIANT)
    else:
        reward, track, transp, effort, jerk, velocity, force_diff = compute_reward_terms(
            pos_error=pos_error_m,
            velocity_error=velocity_error_mps,
            transparency_error=transparency_error_w,
            force_diff=force_diff_n,
            u_v=u_v,
            action_delta=action_delta_v,
            variant=REWARD_VARIANT,
        )
        grouped_terms = {
            'track': track,
            'transparency': transp,
            'effort': effort,
            'jerk': jerk,
            'velocity': velocity,
            'force_diff': force_diff,
        }
        custom_terms = {}
    row = {
        'scenario': name,
        'reward': round(float(reward), 4),
    }
    for key, value in grouped_terms.items():
        if abs(float(value)) > 1e-12:
            row[key] = round(float(value), 4)
    for key, value in custom_terms.items():
        row[f'term_{key}'] = round(float(value), 4)
    return row

sanity_rows = [
    reward_case('good tracking', 0.002, 1.0, u_v=0.5),
    reward_case('good tracking + changed transparency input', 0.002, 15.0, u_v=0.5),
    reward_case('poor tracking', 0.040, 1.0, u_v=0.5),
    reward_case('large action change', 0.002, 1.0, u_v=5.0, action_delta_v=5.0),
]
show_rows(sanity_rows, title='Reward sanity check', max_rows=10)


## 7. Build Run Command

This writes a documentation copy of the specs, builds the command, and shows the exact run configuration. Training uses the named built-in state and reward variants shown below.


In [ ]:
def signal_option(name, *, waveform, amp, bias, omega, phase):
    return {
        'name': str(name),
        'force_waveform': str(waveform),
        'force_amp': float(amp),
        'force_bias': float(bias),
        'force_freq_rad': float(omega),
        'force_phase': float(phase),
    }

TRAIN_SIGNAL_OPTIONS = []
for waveform in ['sine', 'multisine']:
    for amp in [3.0, 5.0, 7.0]:
        for bias in [12.0, 15.0, 18.0]:
            for omega in [6.0]:
                for phase in [0.0, 0.5 * np.pi, np.pi, 1.5 * np.pi]:
                    TRAIN_SIGNAL_OPTIONS.append(signal_option(
                        f'train_{waveform}_a{amp:g}_b{bias:g}_w{omega:g}_p{phase:.2f}',
                        waveform=waveform,
                        amp=amp,
                        bias=bias,
                        omega=omega,
                        phase=phase,
                    ))

EVAL_SIGNAL_OPTIONS = []
for waveform in ['sine', 'multisine']:
    for amp in [4.0, 6.0]:
        for bias in [13.5, 16.5]:
            for omega in [6.0]:
                for phase in [0.25 * np.pi, 0.75 * np.pi, 1.25 * np.pi, 1.75 * np.pi]:
                    EVAL_SIGNAL_OPTIONS.append(signal_option(
                        f'eval_{waveform}_a{amp:g}_b{bias:g}_w{omega:g}_p{phase:.2f}',
                        waveform=waveform,
                        amp=amp,
                        bias=bias,
                        omega=omega,
                        phase=phase,
                    ))
EVAL_SIGNAL_OPTIONS = EVAL_SIGNAL_OPTIONS[:int(CFG['test_episodes'])]

RUN_SPEC = {
    'cfg': CFG,
    'state': STATE_SPEC,
    'reward': REWARD_SPEC,
    'train_signals': TRAIN_SIGNAL_OPTIONS,
    'eval_signals': EVAL_SIGNAL_OPTIONS,
}
SPEC_HASH = short_hash(RUN_SPEC)
CFG['study_name'] = FINAL_RESULTS_SUBFOLDER
SPEC_BASENAME = f"{CFG['study_name']}_{SPEC_HASH}"

SPEC_DIR = PG_RESULTS / 'specs'
STATE_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_state.json"
REWARD_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_reward.json"
TRAIN_SIGNAL_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_train_signals.json"
EVAL_SIGNAL_SPEC_PATH = SPEC_DIR / f"{SPEC_BASENAME}_eval_signals.json"
save_json(STATE_SPEC_PATH, STATE_SPEC)
save_json(REWARD_SPEC_PATH, REWARD_SPEC)
save_json(TRAIN_SIGNAL_SPEC_PATH, {'signals': TRAIN_SIGNAL_OPTIONS})
save_json(EVAL_SIGNAL_SPEC_PATH, {'signals': EVAL_SIGNAL_OPTIONS})

RUN_STUDY_NAME = CFG['study_name']
STEPS_PER_EPISODE = max(1, int(round(CFG['episode_duration_s'] / float(cfg.RL_DT))))
EPISODE_DERIVED_TIMESTEPS = int(CFG['train_episodes'] * STEPS_PER_EPISODE)
TRAIN_TIMESTEPS = int(CFG['total_timesteps'] or EPISODE_DERIVED_TIMESTEPS)
PPO_N_STEPS = int(CFG['ppo_n_steps'])
PPO_BATCH_SIZE = int(CFG['ppo_batch_size'])
PPO_N_EPOCHS = int(CFG['ppo_n_epochs'])
PPO_DEVICE = str(CFG['ppo_device'])
PPO_ROLLOUT_TIMESTEPS = int(CFG['parallel_envs'] * PPO_N_STEPS)
PPO_ACTUAL_TRAIN_TIMESTEPS = int(((TRAIN_TIMESTEPS + PPO_ROLLOUT_TIMESTEPS - 1) // PPO_ROLLOUT_TIMESTEPS) * PPO_ROLLOUT_TIMESTEPS)

RUN_ROOTS = {
    FE_KEY: policy_gradient_suite_root(FE_MODE, RUN_STUDY_NAME) / '00b' / RUN_DIR
}

CMD = [
    str(PYTHON),
    '-m',
    'TeleopWithRL.matlab_literal_env.policy_gradient_experiments.run_policy_gradient_experiments',
    '--algo', ALGO_KEY,
    '--study-name', RUN_STUDY_NAME,
    '--env-mode', CFG['env_mode'],
    '--episode-duration', str(CFG['episode_duration_s']),
    '--env-switch-time', str(CFG['env_switch_time_s']),
    '--fe-mode', FE_MODE,
    '--reset-position-mode', CFG['reset_position_mode'],
    '--stroke-limit-mode', CFG['stroke_limit_mode'],
    '--force-amp', str(CFG['force_amp_N']),
    '--force-bias', str(CFG['force_bias_N']),
    '--force-freq-rad', str(CFG['force_freq_rad_s']),
    '--force-phase', str(CFG['force_phase_rad']),
    '--force-waveform', CFG['force_waveform'],
    '--reward-variant', REWARD_VARIANT.name,
    '--state-variant', STATE_VARIANT.name,
    '--train-reset-options-json', str(TRAIN_SIGNAL_SPEC_PATH),
    '--eval-reset-options-json', str(EVAL_SIGNAL_SPEC_PATH),
    '--train-episodes', str(CFG['train_episodes']),
    '--total-timesteps', str(TRAIN_TIMESTEPS),
    '--parallel-envs', str(CFG['parallel_envs']),
    '--vec-env', CFG['vec_env'],
    '--ppo-n-steps', str(PPO_N_STEPS),
    '--ppo-batch-size', str(PPO_BATCH_SIZE),
    '--ppo-n-epochs', str(PPO_N_EPOCHS),
    '--ppo-device', PPO_DEVICE,
    '--eval-every-episodes', str(CFG['eval_every_episodes']),
    '--test-episodes', str(CFG['test_episodes']),
    '--seed', str(CFG['seed']),
    '--parallel-workers', str(CFG['parallel_workers']),
    '--worker-torch-threads', str(CFG['worker_torch_threads']),
]
if CFG['skip_existing']:
    CMD.append('--skip-existing')

show_rows(
    [{
        'algo': ALGO_LABEL,
        'study_name': RUN_STUDY_NAME,
        'final_results_subfolder': FINAL_RESULTS_SUBFOLDER,
        'spec_hash': SPEC_HASH,
        'reward_variant': REWARD_VARIANT.name,
        'state_variant': STATE_VARIANT.name,
        'state_features': ', '.join(STATE_VARIANT.feature_names),
        'episode_duration_s': CFG['episode_duration_s'],
        'env_switch_time_s': CFG['env_switch_time_s'],
        'stroke_limit_mode': CFG['stroke_limit_mode'],
        'force_amp_N': CFG['force_amp_N'],
        'force_bias_N': CFG['force_bias_N'],
        'force_freq_rad_s': CFG['force_freq_rad_s'],
        'train_signal_count': len(TRAIN_SIGNAL_OPTIONS),
        'eval_signal_count': len(EVAL_SIGNAL_OPTIONS),
        'train_episodes': CFG['train_episodes'],
        'steps_per_episode': STEPS_PER_EPISODE,
        'episode_derived_timesteps': EPISODE_DERIVED_TIMESTEPS,
        'train_timesteps': TRAIN_TIMESTEPS,
        'ppo_n_steps': PPO_N_STEPS,
        'ppo_batch_size': PPO_BATCH_SIZE,
        'ppo_n_epochs': PPO_N_EPOCHS,
        'ppo_device': PPO_DEVICE,
        'ppo_rollout_timesteps': PPO_ROLLOUT_TIMESTEPS,
        'ppo_actual_train_timesteps': PPO_ACTUAL_TRAIN_TIMESTEPS,
        'parallel_envs': CFG['parallel_envs'],
        'vec_env': CFG['vec_env'],
        'eval_every_episodes': CFG['eval_every_episodes'],
        'test_episodes': CFG['test_episodes'],
        'python_executable': str(PYTHON),
        'state_spec_json': str(STATE_SPEC_PATH),
        'reward_spec_json': str(REWARD_SPEC_PATH),
        'train_signal_json': str(TRAIN_SIGNAL_SPEC_PATH),
        'eval_signal_json': str(EVAL_SIGNAL_SPEC_PATH),
        'fe_mode': FE_MODE,
        'run_root': str(RUN_ROOTS[FE_KEY]),
        'model_dir': str(RUN_ROOTS[FE_KEY] / 'm'),
        'plots_dir': str(RUN_ROOTS[FE_KEY] / 'p'),
        'command': subprocess.list2cmdline(CMD),
    }],
    title=f'{ALGO_LABEL} run config',
    max_rows=10,
)
show_rows(TRAIN_SIGNAL_OPTIONS[:16], title='Training input signal pool preview', max_rows=16)
show_rows(EVAL_SIGNAL_OPTIONS[:16], title='Held-out evaluation signal preview', max_rows=16)


## 8. Run Training

This launches the switched-dynamics FE mode only. It can take a while for full `train_episodes`.


In [ ]:
print(subprocess.list2cmdline(CMD))
completed = subprocess.run(CMD, cwd=str(WORKSPACE), check=True)
print(f'Completed with return code {completed.returncode}.')


## 9. Evaluation Config and Model Loading

This section evaluates a frozen trained policy under one-factor-at-a-time stress tests and empirical frequency-response tests. It reuses the existing policy-gradient replica environment, trained model artifact, state variant, reward variant, action bounds, episode timing, and reset options from the notebook run configuration.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import matplotlib
import numpy as np


def activate_notebook_matplotlib() -> str:
    """Prefer inline notebook rendering even if imported package modules selected Agg."""
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    if ip is not None:
        try:
            ip.run_line_magic("matplotlib", "inline")
        except Exception as exc:
            print(f"Could not switch Matplotlib to inline mode: {exc}")
        try:
            from matplotlib_inline.backend_inline import set_matplotlib_formats
            set_matplotlib_formats("retina")
        except Exception:
            pass
    return str(matplotlib.get_backend())


activate_notebook_matplotlib()
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import pandas as pd
except Exception:
    pd = None

from stable_baselines3 import PPO, SAC, TD3
from TeleopWithRL.matlab_literal_env.scripts import run_replica_studies as runner
from TeleopWithRL.matlab_literal_env.simuoriginal_replica import build_saved_simuoriginal_state
from TeleopWithRL.matlab_literal_env.studies.policy_gradient import (
    build_policy_gradient_env_factory,
    get_policy_gradient_reward_variant,
    get_policy_gradient_state_variant,
)

# Some imported training/plotting modules select Agg for file output; switch back here.
MATPLOTLIB_BACKEND = activate_notebook_matplotlib()
try:
    plt.switch_backend("module://matplotlib_inline.backend_inline")
except Exception:
    pass

REPLICA_STATE_INDEX = {
    "Pm1": 0, "Pm2": 1, "xm_dot": 2, "xm": 3,
    "Ps1": 4, "Ps2": 5, "xs_dot": 6, "xs": 7,
    "mL1_dot": 8, "mL2_dot": 9, "x_v": 10, "x_v_dot": 11,
}

EVAL_RUN_ROOT = RUN_ROOTS[FE_KEY]
EVAL_MODEL_PATH = EVAL_RUN_ROOT / "m" / f"{RUN_DIR}_model.zip"
NOMINAL_REPLICA_STATE = build_saved_simuoriginal_state(
    init_position_mode=CFG["reset_position_mode"],
).as_array()

# Frequency values are angular frequencies [rad/s], matching force_freq_rad.
eval_config = {
    "SAVE_RESULTS": False,
    "SHOW_PLOTS": True,
    "PLOT_EACH_SCENARIO": True,
    "PLOT_BODE_TRAJECTORIES": False,
    "RESULTS_DIR": EVAL_RUN_ROOT / "evaluation",
    "MODEL_PATH": EVAL_MODEL_PATH,
    "ALGO_KEY": ALGO_KEY,
    "FE_MODE": FE_MODE,
    "ENV_MODE": CFG["env_mode"],
    "STATE_VARIANT_NAME": STATE_VARIANT.name,
    "REWARD_VARIANT_NAME": REWARD_VARIANT.name,
    "EPISODE_DURATION_S": float(CFG["episode_duration_s"]),
    "CONTACT_TIME_NOMINAL_S": float(CFG["env_switch_time_s"]),
    "RESET_POSITION_MODE": CFG["reset_position_mode"],
    "STROKE_LIMIT_MODE": CFG["stroke_limit_mode"],
    "TERMINATE_ON_ERROR": True,
    "PRE_CONTACT_K_E": float(cfg.SKIN_KE),
    "PRE_CONTACT_B_E": float(cfg.SKIN_BE),
    "K_E_NOMINAL": float(cfg.FAT_KE),
    "B_E_NOMINAL": float(cfg.FAT_BE),
    "FORCE_AMPLITUDE_NOMINAL": float(CFG["force_amp_N"]),
    "FORCE_BIAS_NOMINAL": float(CFG["force_bias_N"]),
    "FORCE_FREQUENCY_NOMINAL_RAD_S": float(CFG["force_freq_rad_s"]),
    "FORCE_PHASE_NOMINAL_RAD": float(CFG["force_phase_rad"]),
    "FORCE_WAVEFORM": CFG["force_waveform"],
    "K_FACTORS": [0.5, 1.0, 1.5],
    "B_FACTORS": [0.5, 1.0, 1.5],
    "CONTACT_TIME_TESTS": sorted({
        max(float(cfg.RL_DT), 0.5 * float(CFG["env_switch_time_s"])),
        float(CFG["env_switch_time_s"]),
        min(float(CFG["episode_duration_s"]) - float(cfg.RL_DT), 1.5 * float(CFG["env_switch_time_s"])),
    }),
    "FORCE_AMPLITUDE_FACTORS": [0.5, 1.0, 1.5],
    "FORCE_FREQUENCY_FACTORS": [0.5, 1.0, 1.5],
    "FORCE_PHASE_TESTS": [0.0, 0.5 * np.pi, np.pi],
    "NOMINAL_REPLICA_STATE": NOMINAL_REPLICA_STATE,
    "INITIAL_CONDITION_TESTS": [
        {"name": "xm_plus_2pct_stroke", "state_delta": {"xm": 0.02 * float(cfg.L_CYL)}},
        {"name": "xs_minus_2pct_stroke", "state_delta": {"xs": -0.02 * float(cfg.L_CYL)}},
    ],
    "ENABLE_SENSOR_NOISE": False,
    "SENSOR_NOISE_STDS": [],  # TODO: add observation-space std values for noisy-policy robustness tests.
    "SETTLING_ERROR_THRESHOLD_M": 0.02 * float(cfg.L_CYL),
    "SETTLING_WINDOW_S": 1.0,
    "SATURATION_TOL_FRACTION": 0.02,
    "TRANSPARENCY_RATIO_EPS_M": 1e-9,
    "PLOT_TRANSPARENCY_RATIO_EACH_SCENARIO": False,
    "BODE_FREQUENCIES": [0.5, 1.0, 2.0, 4.0, 6.0, 8.0, 10.0],
    "BODE_FORCE_AMPLITUDE": float(CFG["force_amp_N"]),
    "BODE_FORCE_PHASE": float(CFG["force_phase_rad"]),
    "BODE_TRANSIENT_S": max(2.0, 0.25 * float(CFG["episode_duration_s"])),
    "BODE_ENV_K_E": float(cfg.FAT_KE),
    "BODE_ENV_B_E": float(cfg.FAT_BE),
    "SEED": int(CFG["seed"]),
}


def safe_filename(text: str) -> str:
    """Return a filesystem-safe stem for scenario artifacts."""
    return "".join(ch if ch.isalnum() or ch in ("-", "_") else "_" for ch in str(text)).strip("_") or "scenario"


def evaluation_dirs(config: dict[str, Any]) -> dict[str, Path]:
    """Create optional evaluation output directories when saving is enabled."""
    root = Path(config["RESULTS_DIR"])
    dirs = {
        "root": root,
        "trajectories": root / "trajectories",
        "metrics": root / "metrics",
        "figures": root / "figures",
        "bode": root / "bode",
    }
    if bool(config.get("SAVE_RESULTS", False)):
        for path in dirs.values():
            path.mkdir(parents=True, exist_ok=True)
    return dirs


def build_eval_env_kwargs(config: dict[str, Any]) -> dict[str, Any]:
    """Build env kwargs through the existing canonical helper."""
    class Args:
        pass

    args = Args()
    args.episode_duration = float(config["EPISODE_DURATION_S"])
    args.env_switch_time = float(config["CONTACT_TIME_NOMINAL_S"])
    args.force_amp = float(config["FORCE_AMPLITUDE_NOMINAL"])
    args.force_bias = float(config["FORCE_BIAS_NOMINAL"])
    args.force_freq_rad = float(config["FORCE_FREQUENCY_NOMINAL_RAD_S"])
    args.force_freq = float(config["FORCE_FREQUENCY_NOMINAL_RAD_S"]) / (2.0 * np.pi)
    args.force_phase = float(config["FORCE_PHASE_NOMINAL_RAD"])
    args.force_waveform = str(config["FORCE_WAVEFORM"])
    args.fe_mode = str(config["FE_MODE"])
    args.reset_position_mode = str(config["RESET_POSITION_MODE"])
    args.stroke_limit_mode = str(config["STROKE_LIMIT_MODE"])
    args.disable_terminate_on_error = not bool(config.get("TERMINATE_ON_ERROR", True))
    args.disable_stroke_limit = False
    args.legacy_baseline_env = False
    args.action_levels = None
    return runner._canonical_env_kwargs(args)


def make_eval_env(config: dict[str, Any]):
    """Create one non-vectorized policy-gradient evaluation environment."""
    state_variant = get_policy_gradient_state_variant(str(config["STATE_VARIANT_NAME"]))
    reward_variant = get_policy_gradient_reward_variant(str(config["REWARD_VARIANT_NAME"]))
    env_factory = build_policy_gradient_env_factory(
        algo=str(config["ALGO_KEY"]),
        env_mode=str(config["ENV_MODE"]),
        env_kwargs=build_eval_env_kwargs(config),
        reward_variant=reward_variant,
        state_variant=state_variant,
    )
    return env_factory()


def load_eval_policy(config: dict[str, Any]):
    """Load the trained SB3 policy using the notebook's saved model path."""
    model_path = Path(config["MODEL_PATH"])
    if not model_path.exists():
        raise FileNotFoundError(f"Missing trained model: {model_path}")
    model_cls = {
        "ppo_continuous": PPO,
        "ppo_discrete": PPO,
        "sac": SAC,
        "td3": TD3,
    }.get(str(config["ALGO_KEY"]))
    if model_cls is None:
        raise KeyError(f"Unsupported policy-gradient algo for eval loading: {config['ALGO_KEY']}")
    return model_cls.load(str(model_path), device="auto")


print("Matplotlib backend:", matplotlib.get_backend())
print("Evaluation model path:", EVAL_MODEL_PATH)
print("Evaluation results dir:", eval_config["RESULTS_DIR"])


## 10. Evaluation Scenarios

The scenario builder creates one-factor-at-a-time tests. `contact_time` maps to the existing environment switch/contact timing. `K_e` and `B_e` are applied to the post-switch/contact environment while the pre-contact values stay nominal.

In [ ]:
@dataclass
class EvaluationScenario:
    """One frozen-policy rollout condition for stress testing."""
    name: str
    group: str
    K_e: float
    B_e: float
    force_amplitude: float
    force_frequency: float  # angular frequency [rad/s]
    force_phase: float
    contact_time: float
    initial_state: Optional[np.ndarray] = None
    sensor_noise_std: float = 0.0
    force_bias: Optional[float] = None
    force_waveform: Optional[str] = None


def _is_nominal(value: float, nominal: float, *, rtol: float = 1e-9, atol: float = 1e-12) -> bool:
    return bool(np.isclose(float(value), float(nominal), rtol=rtol, atol=atol))


def _state_from_delta(config: dict[str, Any], state_delta: dict[str, float]) -> np.ndarray:
    """Apply named perturbations to the replica state vector used by the environment."""
    state = np.asarray(config["NOMINAL_REPLICA_STATE"], dtype=np.float64).copy()
    for key, delta in dict(state_delta).items():
        if key not in REPLICA_STATE_INDEX:
            raise KeyError(f"Unknown replica state key for initial-condition test: {key}")
        state[REPLICA_STATE_INDEX[key]] += float(delta)
    return state


def _base_scenario(config: dict[str, Any], *, name: str, group: str, **overrides) -> EvaluationScenario:
    payload = dict(
        name=name,
        group=group,
        K_e=float(config["K_E_NOMINAL"]),
        B_e=float(config["B_E_NOMINAL"]),
        force_amplitude=float(config["FORCE_AMPLITUDE_NOMINAL"]),
        force_frequency=float(config["FORCE_FREQUENCY_NOMINAL_RAD_S"]),
        force_phase=float(config["FORCE_PHASE_NOMINAL_RAD"]),
        contact_time=float(config["CONTACT_TIME_NOMINAL_S"]),
        force_bias=float(config["FORCE_BIAS_NOMINAL"]),
        force_waveform=str(config["FORCE_WAVEFORM"]),
    )
    payload.update(overrides)
    return EvaluationScenario(**payload)


def build_evaluation_scenarios(config: dict[str, Any]) -> list[EvaluationScenario]:
    """Build nominal plus one-factor-at-a-time stress scenarios."""
    K0 = float(config["K_E_NOMINAL"])
    B0 = float(config["B_E_NOMINAL"])
    A0 = float(config["FORCE_AMPLITUDE_NOMINAL"])
    w0 = float(config["FORCE_FREQUENCY_NOMINAL_RAD_S"])
    p0 = float(config["FORCE_PHASE_NOMINAL_RAD"])
    c0 = float(config["CONTACT_TIME_NOMINAL_S"])
    scenarios = [_base_scenario(config, name="nominal", group="nominal")]

    for factor in config.get("K_FACTORS", []):
        if not _is_nominal(factor, 1.0):
            scenarios.append(_base_scenario(config, name=f"env_K_x{factor:g}".replace(".", "p"), group="environment_variation", K_e=K0 * float(factor)))
    for factor in config.get("B_FACTORS", []):
        if not _is_nominal(factor, 1.0):
            scenarios.append(_base_scenario(config, name=f"env_B_x{factor:g}".replace(".", "p"), group="environment_variation", B_e=B0 * float(factor)))
    for contact_time in config.get("CONTACT_TIME_TESTS", []):
        if not _is_nominal(contact_time, c0):
            scenarios.append(_base_scenario(config, name=f"contact_t{float(contact_time):g}s".replace(".", "p"), group="contact_time_variation", contact_time=float(contact_time)))
    for factor in config.get("FORCE_AMPLITUDE_FACTORS", []):
        if not _is_nominal(factor, 1.0):
            scenarios.append(_base_scenario(config, name=f"force_amp_x{factor:g}".replace(".", "p"), group="force_amplitude_variation", force_amplitude=A0 * float(factor)))
    for factor in config.get("FORCE_FREQUENCY_FACTORS", []):
        if not _is_nominal(factor, 1.0):
            scenarios.append(_base_scenario(config, name=f"force_freq_x{factor:g}".replace(".", "p"), group="force_frequency_variation", force_frequency=w0 * float(factor)))
    for phase in config.get("FORCE_PHASE_TESTS", []):
        if not _is_nominal(phase, p0):
            scenarios.append(_base_scenario(config, name=f"force_phase_{float(phase):.2f}rad".replace(".", "p"), group="force_phase_variation", force_phase=float(phase)))
    for row in config.get("INITIAL_CONDITION_TESTS", []):
        scenarios.append(_base_scenario(config, name=str(row["name"]), group="initial_condition_variation", initial_state=_state_from_delta(config, row.get("state_delta", {}))))
    if bool(config.get("ENABLE_SENSOR_NOISE", False)):
        for std in config.get("SENSOR_NOISE_STDS", []):
            scenarios.append(_base_scenario(config, name=f"sensor_noise_{float(std):g}".replace(".", "p"), group="sensor_noise_variation", sensor_noise_std=float(std)))
    return scenarios


def scenario_reset_options(scenario: EvaluationScenario, config: dict[str, Any]) -> dict[str, Any]:
    """Translate a scenario into backward-compatible env.reset(options=...) values."""
    options = {
        "name": scenario.name,
        "force_amp": float(scenario.force_amplitude),
        "force_bias": float(config["FORCE_BIAS_NOMINAL"] if scenario.force_bias is None else scenario.force_bias),
        "force_freq_rad": float(scenario.force_frequency),
        "force_phase": float(scenario.force_phase),
        "force_waveform": str(config["FORCE_WAVEFORM"] if scenario.force_waveform is None else scenario.force_waveform),
        "env_switch_time": float(scenario.contact_time),
        "pre_switch_Ke": float(config["PRE_CONTACT_K_E"]),
        "pre_switch_Be": float(config["PRE_CONTACT_B_E"]),
        "post_switch_Ke": float(scenario.K_e),
        "post_switch_Be": float(scenario.B_e),
        "fe_mode": str(config["FE_MODE"]),
        "reset_position_mode": str(config["RESET_POSITION_MODE"]),
        "stroke_limit_mode": str(config["STROKE_LIMIT_MODE"]),
    }
    if scenario.initial_state is not None:
        options["initial_state"] = np.asarray(scenario.initial_state, dtype=np.float64)
    return options


scenarios_preview = build_evaluation_scenarios(eval_config)
print(f"Built {len(scenarios_preview)} normal evaluation scenarios.")
for scenario in scenarios_preview[:8]:
    print(f"- {scenario.group}: {scenario.name}")


## 11. Evaluation Runners, Metrics, and Plots

These functions run frozen policies with deterministic actions by default, collect trajectories from the existing environment history, compute tracking/transparency/control metrics, and display plots directly in the notebook.

In [ ]:
def _arr(trajectory: dict[str, Any], key: str, dtype=np.float64) -> np.ndarray:
    if key not in trajectory:
        return np.asarray([], dtype=dtype)
    try:
        return np.asarray(trajectory[key], dtype=dtype)
    except (TypeError, ValueError):
        return np.asarray(trajectory[key], dtype=object)


def _wrap_to_pi(angle: float) -> float:
    return float((float(angle) + np.pi) % (2.0 * np.pi) - np.pi)


def _show_or_close(fig, show: bool = True):
    """Show figures in notebooks; fall back to display(fig) if the backend is non-interactive Agg."""
    if not show:
        plt.close(fig)
        return
    backend = str(matplotlib.get_backend()).lower()
    if "agg" in backend and "inline" not in backend:
        try:
            display(fig)
        finally:
            plt.close(fig)
        return
    plt.show()


def fit_sine_amplitude_phase(t: np.ndarray, y: np.ndarray, omega: float) -> tuple[float, float]:
    """Fit y ~= a*sin(wt) + b*cos(wt) + c and return amplitude and phase."""
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(t) & np.isfinite(y)
    if np.count_nonzero(mask) < 6 or abs(float(omega)) <= 1e-12:
        return np.nan, np.nan
    tt = t[mask]
    yy = y[mask]
    X = np.column_stack([np.sin(float(omega) * tt), np.cos(float(omega) * tt), np.ones_like(tt)])
    coeffs, *_ = np.linalg.lstsq(X, yy, rcond=None)
    sin_coeff, cos_coeff = float(coeffs[0]), float(coeffs[1])
    return float(np.hypot(sin_coeff, cos_coeff)), float(np.arctan2(cos_coeff, sin_coeff))


def estimate_gain_phase(t: np.ndarray, x_m: np.ndarray, x_s: np.ndarray, omega: float, transient_s: float = 0.0):
    """Estimate slave/master magnitude ratio and phase lag using sinusoidal fits."""
    n = min(np.asarray(t).size, np.asarray(x_m).size, np.asarray(x_s).size)
    if n < 6:
        return np.nan, np.nan, np.nan, np.nan
    t = np.asarray(t, dtype=float)[:n]
    x_m = np.asarray(x_m, dtype=float)[:n]
    x_s = np.asarray(x_s, dtype=float)[:n]
    mask = t >= float(transient_s)
    if np.count_nonzero(mask) < 6:
        mask = np.ones_like(t, dtype=bool)
    amp_m, phase_m = fit_sine_amplitude_phase(t[mask], x_m[mask], omega)
    amp_s, phase_s = fit_sine_amplitude_phase(t[mask], x_s[mask], omega)
    gain = float(amp_s / amp_m) if np.isfinite(amp_m) and amp_m > 1e-12 else np.nan
    gain_db = float(20.0 * np.log10(max(gain, 1e-12))) if np.isfinite(gain) else np.nan
    phase_lag = _wrap_to_pi(phase_s - phase_m) if np.isfinite(phase_s) and np.isfinite(phase_m) else np.nan
    return gain, gain_db, phase_lag, amp_m


def settling_time_after_contact(t: np.ndarray, error: np.ndarray, contact_time: float, threshold: float, window_s: float) -> float:
    """First time after contact where abs(error) remains below threshold for a full window."""
    n = min(np.asarray(t).size, np.asarray(error).size)
    if n == 0:
        return np.nan
    t = np.asarray(t, dtype=float)[:n]
    error = np.asarray(error, dtype=float)[:n]
    after = np.flatnonzero(t >= float(contact_time))
    if after.size == 0:
        return np.nan
    dt = float(np.nanmedian(np.diff(t))) if t.size > 1 else float(cfg.RL_DT)
    window_steps = max(1, int(round(float(window_s) / max(dt, 1e-12))))
    below = np.abs(error) <= float(threshold)
    for idx in after:
        end = min(n, idx + window_steps)
        if end - idx < window_steps:
            break
        if bool(np.all(below[idx:end])):
            return float(t[idx] - float(contact_time))
    return np.nan


def evaluate_policy(policy, env, scenario: EvaluationScenario, config: dict[str, Any], deterministic: bool = True) -> dict[str, Any]:
    """Run one frozen-policy rollout and return the full trajectory dictionary."""
    options = scenario_reset_options(scenario, config)
    obs, info = env.reset(seed=int(config.get("SEED", 0)), options=options)
    rng = np.random.default_rng(int(config.get("SEED", 0)) + abs(hash(scenario.name)) % 1_000_000)
    terminated = truncated = False
    rewards = []
    while not bool(terminated or truncated):
        policy_obs = np.asarray(obs, dtype=np.float32)
        if float(scenario.sensor_noise_std) > 0.0:
            policy_obs = policy_obs + rng.normal(0.0, float(scenario.sensor_noise_std), size=policy_obs.shape).astype(np.float32)
        if hasattr(policy, "predict"):
            action, _ = policy.predict(policy_obs, deterministic=deterministic)
        else:
            action = policy(policy_obs)
        obs, reward, terminated, truncated, info = env.step(action)
        rewards.append(float(reward))
    history = env.render() or {}
    trajectory = {key: np.asarray(value) for key, value in history.items()}
    trajectory.update({
        "scenario_name": scenario.name,
        "scenario_group": scenario.group,
        "scenario": scenario,
        "reset_options": options,
        "episode_return": float(np.sum(rewards)) if rewards else 0.0,
        "terminated": bool(terminated),
        "truncated": bool(truncated),
    })
    if hasattr(env, "action_space") and hasattr(env.action_space, "low"):
        trajectory["action_low"] = np.asarray(env.action_space.low, dtype=float)
        trajectory["action_high"] = np.asarray(env.action_space.high, dtype=float)
    return trajectory


def _finite_stat(values, reducer) -> float:
    """Apply a reducer after dropping non-finite values."""
    arr = np.asarray(values, dtype=float).reshape(-1)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    return float(reducer(arr))


def position_transparency_ratio(x_m: np.ndarray, x_s: np.ndarray, eps_m: float = 1e-9) -> np.ndarray:
    """Compute the requested position transparency ratio x_m / x_s safely."""
    n = min(np.asarray(x_m).size, np.asarray(x_s).size)
    ratio = np.full(n, np.nan, dtype=float)
    if n == 0:
        return ratio
    xm = np.asarray(x_m, dtype=float)[:n]
    xs = np.asarray(x_s, dtype=float)[:n]
    valid = np.isfinite(xm) & np.isfinite(xs) & (np.abs(xs) > float(eps_m))
    ratio[valid] = xm[valid] / xs[valid]
    return ratio


def compute_metrics(trajectory: dict[str, Any], scenario: EvaluationScenario, config: dict[str, Any]) -> dict[str, Any]:
    """Compute tracking, adaptation, control, saturation, and frequency-response metrics."""
    t = _arr(trajectory, "time")
    x_m = _arr(trajectory, "x_m")
    x_s = _arr(trajectory, "x_s")
    u = _arr(trajectory, "u_v")
    invalid_state = _arr(trajectory, "invalid_state", dtype=bool)
    tracking_fail = _arr(trajectory, "tracking_error_fail", dtype=bool)
    n = min(t.size, x_m.size, x_s.size)
    if n == 0:
        error = np.asarray([], dtype=float)
        dt = float(cfg.RL_DT)
    else:
        t = t[:n]
        x_m = x_m[:n]
        x_s = x_s[:n]
        error = x_m - x_s
        dt = float(np.nanmedian(np.diff(t))) if t.size > 1 else float(cfg.RL_DT)
    after = t >= float(scenario.contact_time) if t.size else np.asarray([], dtype=bool)
    post_error = error[after] if error.size and after.size else np.asarray([], dtype=float)
    xm_over_xs = position_transparency_ratio(x_m, x_s, eps_m=float(config.get("TRANSPARENCY_RATIO_EPS_M", 1e-9))) if n else np.asarray([], dtype=float)
    post_xm_over_xs = xm_over_xs[after] if xm_over_xs.size and after.size else np.asarray([], dtype=float)
    xm_over_xs_error = xm_over_xs - 1.0 if xm_over_xs.size else np.asarray([], dtype=float)
    post_xm_over_xs_error = post_xm_over_xs - 1.0 if post_xm_over_xs.size else np.asarray([], dtype=float)
    u = u[: min(u.size, t.size)] if u.size and t.size else u
    du = np.diff(u) if u.size >= 2 else np.asarray([], dtype=float)

    action_low = np.asarray(trajectory.get("action_low", [np.nan]), dtype=float).reshape(-1)
    action_high = np.asarray(trajectory.get("action_high", [np.nan]), dtype=float).reshape(-1)
    low = float(action_low[0]) if action_low.size else np.nan
    high = float(action_high[0]) if action_high.size else np.nan
    sat_pct = np.nan
    if u.size and np.isfinite(low) and np.isfinite(high):
        tol = float(config["SATURATION_TOL_FRACTION"]) * max(abs(low), abs(high), 1e-12)
        sat_pct = 100.0 * float(np.mean((u <= low + tol) | (u >= high - tol)))
    gain, gain_db, phase_lag, _ = estimate_gain_phase(
        t, x_m, x_s, float(scenario.force_frequency),
        transient_s=max(float(scenario.contact_time), float(config.get("BODE_TRANSIENT_S", 0.0))),
    )
    reasons = _arr(trajectory, "termination_reason", dtype=object)
    termination_reason = str(reasons[-1]) if reasons.size else ""
    failure = bool(
        (invalid_state.size and np.any(invalid_state))
        or (tracking_fail.size and np.any(tracking_fail))
        or (trajectory.get("terminated", False) and termination_reason not in {"", "max_steps"})
    )
    return {
        "scenario": scenario.name,
        "group": scenario.group,
        "K_e": float(scenario.K_e),
        "B_e": float(scenario.B_e),
        "force_amplitude": float(scenario.force_amplitude),
        "force_frequency_rad_s": float(scenario.force_frequency),
        "force_phase_rad": float(scenario.force_phase),
        "contact_time_s": float(scenario.contact_time),
        "return": float(trajectory.get("episode_return", 0.0)),
        "rms_tracking_error_m": float(np.sqrt(np.nanmean(error ** 2))) if error.size else np.nan,
        "peak_abs_tracking_error_m": float(np.nanmax(np.abs(error))) if error.size else np.nan,
        "post_contact_rms_error_m": float(np.sqrt(np.nanmean(post_error ** 2))) if post_error.size else np.nan,
        "post_contact_peak_error_m": float(np.nanmax(np.abs(post_error))) if post_error.size else np.nan,
        "settling_time_after_contact_s": settling_time_after_contact(t, error, float(scenario.contact_time), float(config["SETTLING_ERROR_THRESHOLD_M"]), float(config["SETTLING_WINDOW_S"])),
        "control_energy": float(np.nansum(u ** 2) * dt) if u.size else np.nan,
        "control_smoothness": float(np.nansum(du ** 2) * dt) if du.size else np.nan,
        "valve_saturation_pct": sat_pct,
        "failure_flag": float(failure),
        "transparency_xm_over_xs_mean": _finite_stat(xm_over_xs, np.mean),
        "transparency_xm_over_xs_std": _finite_stat(xm_over_xs, np.std),
        "transparency_xm_over_xs_rmse_from_1": _finite_stat(xm_over_xs_error, lambda arr: np.sqrt(np.mean(arr ** 2))),
        "transparency_xm_over_xs_peak_abs_error_from_1": _finite_stat(xm_over_xs_error, lambda arr: np.max(np.abs(arr))),
        "post_contact_transparency_xm_over_xs_mean": _finite_stat(post_xm_over_xs, np.mean),
        "post_contact_transparency_xm_over_xs_rmse_from_1": _finite_stat(post_xm_over_xs_error, lambda arr: np.sqrt(np.mean(arr ** 2))),
        "transparency_xm_over_xs_valid_pct": 100.0 * float(np.mean(np.isfinite(xm_over_xs))) if xm_over_xs.size else np.nan,
        "position_magnitude_ratio": gain,
        "position_gain_db": gain_db,
        "phase_lag_rad": phase_lag,
        "phase_lag_deg": float(np.rad2deg(phase_lag)) if np.isfinite(phase_lag) else np.nan,
        "steps": int(t.size),
        "termination_reason": termination_reason,
    }


def _figure_path(save_path: Optional[Path], suffix: str) -> Optional[Path]:
    if save_path is None:
        return None
    save_path = Path(save_path)
    if save_path.suffix:
        return save_path
    save_path.mkdir(parents=True, exist_ok=True)
    return save_path / suffix


def plot_trajectory(trajectory: dict[str, Any], scenario: EvaluationScenario, save_path: Optional[Path] = None, show: bool = True):
    """Plot master/slave position versus time."""
    t = _arr(trajectory, "time")
    x_m = _arr(trajectory, "x_m")
    x_s = _arr(trajectory, "x_s")
    n = min(t.size, x_m.size, x_s.size)
    if n == 0:
        print(f"No position data for {scenario.name}")
        return None
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t[:n], 1000.0 * x_m[:n], label="x_m")
    ax.plot(t[:n], 1000.0 * x_s[:n], label="x_s")
    ax.axvline(float(scenario.contact_time), color="0.35", ls="--", lw=1.0, label="contact/switch")
    ax.set_title(f"{scenario.name}: master/slave position")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Position [mm]")
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = _figure_path(save_path, f"{safe_filename(scenario.name)}_positions.png")
    if out is not None:
        fig.savefig(out, dpi=150, bbox_inches="tight")
    _show_or_close(fig, show=show)
    return fig


def plot_error_and_action(trajectory: dict[str, Any], scenario: EvaluationScenario, save_path: Optional[Path] = None, show: bool = True):
    """Plot tracking error and valve/action input versus time."""
    t = _arr(trajectory, "time")
    x_m = _arr(trajectory, "x_m")
    x_s = _arr(trajectory, "x_s")
    u = _arr(trajectory, "u_v")
    n = min(t.size, x_m.size, x_s.size)
    if n == 0:
        print(f"No error data for {scenario.name}")
        return None
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, constrained_layout=True)
    axes[0].plot(t[:n], 1000.0 * (x_m[:n] - x_s[:n]), color="tab:purple")
    axes[0].axhline(0.0, color="0.35", lw=0.8)
    axes[0].axvline(float(scenario.contact_time), color="0.35", ls="--", lw=1.0)
    axes[0].set_title(f"{scenario.name}: tracking error and action")
    axes[0].set_ylabel("x_m - x_s [mm]")
    axes[0].grid(True, alpha=0.3)
    m = min(t.size, u.size)
    if m:
        axes[1].plot(t[:m], u[:m], color="tab:red", label="u_v")
    low = np.asarray(trajectory.get("action_low", []), dtype=float).reshape(-1)
    high = np.asarray(trajectory.get("action_high", []), dtype=float).reshape(-1)
    if low.size and high.size:
        axes[1].axhline(float(low[0]), color="0.35", ls="--", lw=0.8)
        axes[1].axhline(float(high[0]), color="0.35", ls="--", lw=0.8)
    axes[1].axvline(float(scenario.contact_time), color="0.35", ls="--", lw=1.0)
    axes[1].set_xlabel("Time [s]")
    axes[1].set_ylabel("Valve input [V]")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc="upper right")
    out = _figure_path(save_path, f"{safe_filename(scenario.name)}_error_action.png")
    if out is not None:
        fig.savefig(out, dpi=150, bbox_inches="tight")
    _show_or_close(fig, show=show)
    return fig


def plot_transparency_ratio(trajectory: dict[str, Any], scenario: EvaluationScenario, save_path: Optional[Path] = None, show: bool = True):
    """Plot the requested position transparency ratio x_m / x_s versus time."""
    t = _arr(trajectory, "time")
    x_m = _arr(trajectory, "x_m")
    x_s = _arr(trajectory, "x_s")
    n = min(t.size, x_m.size, x_s.size)
    if n == 0:
        print(f"No transparency-ratio data for {scenario.name}")
        return None
    ratio = position_transparency_ratio(x_m[:n], x_s[:n])
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(t[:n], ratio, color="tab:green", lw=1.2, label="x_m / x_s")
    ax.axhline(1.0, color="0.25", ls="--", lw=0.9, label="ideal = 1")
    ax.axvline(float(scenario.contact_time), color="0.35", ls="--", lw=1.0, label="contact/switch")
    ax.set_title(f"{scenario.name}: position transparency ratio")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("x_m / x_s [-]")
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = _figure_path(save_path, f"{safe_filename(scenario.name)}_xm_over_xs.png")
    if out is not None:
        fig.savefig(out, dpi=150, bbox_inches="tight")
    _show_or_close(fig, show=show)
    return fig


def plot_transparency_ratio_across_scenarios(trajectories: dict[str, dict[str, Any]], scenario_names=None, show: bool = True, save_dir: Optional[Path] = None):
    """Overlay x_m / x_s for every evaluated scenario, grouped to keep plots readable."""
    names = list(trajectories.keys()) if scenario_names is None else list(scenario_names)
    items = [(name, trajectories[name]) for name in names if name in trajectories]
    if not items:
        print("No trajectories available for transparency-ratio comparison.")
        return
    save_dir = Path(save_dir) if save_dir is not None else None
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
    groups = []
    for _, trajectory in items:
        group = str(trajectory.get("scenario_group", getattr(trajectory.get("scenario", None), "group", "unknown")))
        if group not in groups:
            groups.append(group)
    for group in groups:
        group_items = [(name, tr) for name, tr in items if str(tr.get("scenario_group", getattr(tr.get("scenario", None), "group", "unknown"))) == group]
        fig, ax = plt.subplots(figsize=(13, 5))
        contact_times = []
        for name, trajectory in group_items:
            scenario = trajectory.get("scenario")
            t = _arr(trajectory, "time")
            x_m = _arr(trajectory, "x_m")
            x_s = _arr(trajectory, "x_s")
            n = min(t.size, x_m.size, x_s.size)
            if n == 0:
                continue
            ratio = position_transparency_ratio(x_m[:n], x_s[:n])
            ax.plot(t[:n], ratio, lw=1.1, alpha=0.9, label=name)
            if scenario is not None:
                contact_times.append(float(scenario.contact_time))
        ax.axhline(1.0, color="0.2", ls="--", lw=0.9, label="ideal = 1")
        for contact_time in sorted(set(round(value, 6) for value in contact_times)):
            ax.axvline(contact_time, color="0.45", ls=":", lw=0.8, alpha=0.7)
        ax.set_title(f"Position transparency ratio x_m / x_s: {group}")
        ax.set_xlabel("Time [s]")
        ax.set_ylabel("x_m / x_s [-]")
        ax.grid(True, alpha=0.3)
        ax.legend(ncol=2, fontsize=8)
        if save_dir is not None:
            fig.savefig(save_dir / f"transparency_xm_over_xs_{safe_filename(group)}.png", dpi=150, bbox_inches="tight")
        _show_or_close(fig, show=show)


def save_trajectory_npz(trajectory: dict[str, Any], out_path: Path) -> None:
    payload = {}
    for key, value in trajectory.items():
        if key in {"scenario", "reset_options"}:
            continue
        arr = np.asarray(value)
        if arr.ndim > 0 and arr.dtype != object:
            payload[key] = arr
    np.savez(out_path, **payload)


def run_evaluation_suite(policy, env_or_factory, scenarios: list[EvaluationScenario], config: dict[str, Any], deterministic: bool = True):
    """Run all normal stress-test scenarios and return trajectories plus a metrics table."""
    dirs = evaluation_dirs(config)
    trajectories = {}
    metric_rows = []
    use_factory = callable(env_or_factory) and not hasattr(env_or_factory, "reset")
    for idx, scenario in enumerate(scenarios, start=1):
        print(f"[{idx}/{len(scenarios)}] evaluating {scenario.group}/{scenario.name}")
        env = env_or_factory() if use_factory else env_or_factory
        try:
            trajectory = evaluate_policy(policy, env, scenario, config, deterministic=deterministic)
        finally:
            if use_factory and hasattr(env, "close"):
                env.close()
        trajectories[scenario.name] = trajectory
        metric_rows.append(compute_metrics(trajectory, scenario, config))
        if bool(config.get("SAVE_RESULTS", False)):
            save_trajectory_npz(trajectory, dirs["trajectories"] / f"{safe_filename(scenario.name)}.npz")
        if bool(config.get("PLOT_EACH_SCENARIO", True)):
            fig_dir = dirs["figures"] if bool(config.get("SAVE_RESULTS", False)) else None
            plot_trajectory(trajectory, scenario, save_path=fig_dir, show=bool(config.get("SHOW_PLOTS", True)))
            plot_error_and_action(trajectory, scenario, save_path=fig_dir, show=bool(config.get("SHOW_PLOTS", True)))
            if bool(config.get("PLOT_TRANSPARENCY_RATIO_EACH_SCENARIO", False)):
                plot_transparency_ratio(trajectory, scenario, save_path=fig_dir, show=bool(config.get("SHOW_PLOTS", True)))
    metrics_df = pd.DataFrame(metric_rows) if pd is not None else metric_rows
    if bool(config.get("SAVE_RESULTS", False)) and pd is not None:
        metrics_df.to_csv(dirs["metrics"] / "evaluation_metrics.csv", index=False)
    return trajectories, metrics_df


def plot_selected_scenarios(trajectories: dict[str, dict[str, Any]], scenario_names=None, show: bool = True, include_transparency: bool = True):
    """Display individual scenario plots from an existing trajectory dictionary."""
    names = list(trajectories.keys()) if scenario_names is None else list(scenario_names)
    for name in names:
        if name not in trajectories:
            print(f"Missing trajectory: {name}")
            continue
        trajectory = trajectories[name]
        scenario = trajectory["scenario"]
        plot_trajectory(trajectory, scenario, show=show)
        plot_error_and_action(trajectory, scenario, show=show)
        if include_transparency:
            plot_transparency_ratio(trajectory, scenario, show=show)


def plot_metrics_summary(metrics_df, show: bool = True, save_dir: Optional[Path] = None):
    """Plot summary bars for key metrics, grouped by scenario group."""
    if pd is None:
        print("Pandas is unavailable; metrics summary plotting expects a DataFrame.")
        return
    metric_names = [
        "rms_tracking_error_m", "peak_abs_tracking_error_m",
        "post_contact_rms_error_m", "post_contact_peak_error_m",
        "settling_time_after_contact_s", "control_energy",
        "control_smoothness", "valve_saturation_pct",
        "failure_flag",
        "transparency_xm_over_xs_mean",
        "transparency_xm_over_xs_rmse_from_1",
        "post_contact_transparency_xm_over_xs_mean",
        "post_contact_transparency_xm_over_xs_rmse_from_1",
        "position_magnitude_ratio", "phase_lag_deg",
    ]
    save_dir = Path(save_dir) if save_dir is not None else None
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
    for group, group_df in metrics_df.groupby("group", sort=False):
        present = [metric for metric in metric_names if metric in group_df.columns]
        rows = int(np.ceil(len(present) / 2))
        fig, axes = plt.subplots(rows, 2, figsize=(14, max(4, 2.8 * rows)), constrained_layout=True)
        axes = np.asarray(axes).reshape(-1)
        for ax, metric in zip(axes, present):
            ax.bar(group_df["scenario"], group_df[metric])
            ax.set_title(metric)
            ax.tick_params(axis="x", labelrotation=45)
            ax.grid(True, axis="y", alpha=0.25)
        for ax in axes[len(present):]:
            ax.axis("off")
        fig.suptitle(f"Metrics summary: {group}", y=1.01)
        if save_dir is not None:
            fig.savefig(save_dir / f"metrics_summary_{safe_filename(group)}.png", dpi=150, bbox_inches="tight")
        _show_or_close(fig, show=show)


def run_empirical_bode(policy, env_or_factory, config: dict[str, Any], deterministic: bool = True):
    """Run empirical frequency-response rollouts for the nonlinear learned policy."""
    dirs = evaluation_dirs(config)
    rows = []
    use_factory = callable(env_or_factory) and not hasattr(env_or_factory, "reset")
    for omega in config.get("BODE_FREQUENCIES", []):
        scenario = _base_scenario(
            config,
            name=f"bode_{float(omega):g}_rad_s".replace(".", "p"),
            group="empirical_bode",
            K_e=float(config["BODE_ENV_K_E"]),
            B_e=float(config["BODE_ENV_B_E"]),
            force_amplitude=float(config["BODE_FORCE_AMPLITUDE"]),
            force_frequency=float(omega),
            force_phase=float(config["BODE_FORCE_PHASE"]),
        )
        print(f"empirical bode: omega={float(omega):g} rad/s")
        env = env_or_factory() if use_factory else env_or_factory
        try:
            trajectory = evaluate_policy(policy, env, scenario, config, deterministic=deterministic)
        finally:
            if use_factory and hasattr(env, "close"):
                env.close()
        t = _arr(trajectory, "time")
        x_m = _arr(trajectory, "x_m")
        x_s = _arr(trajectory, "x_s")
        gain, gain_db, phase_lag, amp_m = estimate_gain_phase(t, x_m, x_s, float(omega), transient_s=float(config["BODE_TRANSIENT_S"]))
        rows.append({
            "frequency_rad_s": float(omega),
            "frequency_hz": float(omega) / (2.0 * np.pi),
            "gain": gain,
            "gain_dB": gain_db,
            "phase_lag_rad": phase_lag,
            "phase_lag_deg": float(np.rad2deg(phase_lag)) if np.isfinite(phase_lag) else np.nan,
            "master_amplitude_m": amp_m,
        })
        if bool(config.get("SAVE_RESULTS", False)) and bool(config.get("PLOT_BODE_TRAJECTORIES", False)):
            save_trajectory_npz(trajectory, dirs["bode"] / f"{safe_filename(scenario.name)}.npz")
    bode_df = pd.DataFrame(rows) if pd is not None else rows
    if bool(config.get("SAVE_RESULTS", False)) and pd is not None:
        bode_df.to_csv(dirs["bode"] / "empirical_bode.csv", index=False)
    return bode_df


def plot_empirical_bode(bode_df, show: bool = True, save_dir: Optional[Path] = None):
    """Display empirical Bode magnitude and phase plots in the notebook."""
    if pd is None:
        print("Pandas is unavailable; empirical Bode plotting expects a DataFrame.")
        return
    if bode_df is None or len(bode_df) == 0:
        print("No Bode rows to plot.")
        return
    save_dir = Path(save_dir) if save_dir is not None else None
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
    x = bode_df["frequency_rad_s"].to_numpy(dtype=float)
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True, constrained_layout=True)
    axes[0].semilogx(x, bode_df["gain_dB"], marker="o")
    axes[0].axhline(0.0, color="0.35", ls="--", lw=0.8)
    axes[0].set_ylabel("Magnitude [dB]")
    axes[0].set_title("Empirical Bode magnitude: x_s / x_m")
    axes[0].grid(True, which="both", alpha=0.3)
    axes[1].semilogx(x, bode_df["phase_lag_deg"], marker="o", color="tab:orange")
    axes[1].axhline(0.0, color="0.35", ls="--", lw=0.8)
    axes[1].set_xlabel("Angular frequency [rad/s]")
    axes[1].set_ylabel("Phase lag [deg]")
    axes[1].set_title("Empirical Bode phase")
    axes[1].grid(True, which="both", alpha=0.3)
    if save_dir is not None:
        fig.savefig(save_dir / "empirical_bode.png", dpi=150, bbox_inches="tight")
    _show_or_close(fig, show=show)


## 12. Run Evaluation Workflow

Run this cell after a trained model exists at the path printed above. It runs the normal stress-test suite, displays/saves metrics and trajectory plots according to `eval_config`, then runs the empirical Bode test.

In [ ]:
# 1. Load policy and create an environment factory.
policy = load_eval_policy(eval_config)
env_factory = lambda: make_eval_env(eval_config)

# 2. Build one-factor-at-a-time scenarios.
scenarios = build_evaluation_scenarios(eval_config)
print(f"Normal evaluation scenarios: {len(scenarios)}")

# 3. Run normal stress tests.
trajectories, metrics_df = run_evaluation_suite(
    policy,
    env_factory,
    scenarios,
    eval_config,
    deterministic=True,
)

# 4. Display metrics table in the notebook.
display(metrics_df)

# 5. Show selected scenario plots again if desired.
# plot_selected_scenarios(trajectories, scenario_names=["nominal"], show=True)

# 6. Show summary plots.
summary_save_dir = evaluation_dirs(eval_config)["figures"] if eval_config["SAVE_RESULTS"] else None
plot_metrics_summary(metrics_df, show=True, save_dir=summary_save_dir)
plot_transparency_ratio_across_scenarios(trajectories, show=True, save_dir=summary_save_dir)

# 7. Run empirical Bode test.
bode_df = run_empirical_bode(policy, env_factory, eval_config, deterministic=True)

# 8. Display Bode table and plots.
display(bode_df)
bode_save_dir = evaluation_dirs(eval_config)["bode"] if eval_config["SAVE_RESULTS"] else None
plot_empirical_bode(bode_df, show=True, save_dir=bode_save_dir)
